# 05 · Síntese, resposta e recomendações

**Projeto Integrador** · Jeanderson Soares Mingorança  
**Fase do roadmap:** 5 · Escopo: somente a base comum (DA-017)

Reúne as evidências dos notebooks 03 e 04 em uma tabela-síntese, responde à pergunta e registra recomendações e limitações. As recomendações estão explicadas em linguagem acessível em `docs/RECOMENDACOES_EXPLICADAS.md`; o produto de comunicação está em `entrega/JeandersonMingoranca_RelatorioUmaFolha.pdf`.

## 1. Ambiente

In [1]:
from pathlib import Path
import sys

# Localiza a raiz do projeto (pasta com pyproject.toml) a partir do diretório atual,
# para que o notebook funcione aberto de notebooks/ ou da raiz.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

PASTA_TABELAS = RAIZ / "resultados" / "tabelas"
PASTA_FIGURAS = RAIZ / "resultados" / "figuras"
PASTA_PROCESSADOS = RAIZ / "dados" / "processados"
for pasta in (PASTA_TABELAS, PASTA_FIGURAS, PASTA_PROCESSADOS):
    pasta.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| numpy:", np.__version__, "| matplotlib:", matplotlib.__version__)
print("Raiz do projeto:", RAIZ.name)

Python: 3.12.10
pandas: 3.0.5 | numpy: 2.5.3 | matplotlib: 3.11.2
Raiz do projeto: JeandersonMingoranca_ProjetoIntegrador


## 2. Tabela-síntese: diferença ajustada de 2025 em relação a 2023

In [2]:
coef = pd.read_csv(PASTA_TABELAS / "04_coeficientes_principal.csv")
estab = pd.read_csv(PASTA_TABELAS / "04_estabilidade_cenarios.csv")
c25 = coef[coef["ano"] == 2025].merge(estab[["nivel", "indicador", "robusto"]], on=["nivel", "indicador"])

escala = {"indice_vale": (1, ""), "carga_residual_min": (1000, " GW"), "rampa_tarde": (1000, " GW"),
          "maior_rampa_horaria": (1000, " GW/h"), "horas_residual_baixa": (1, " h/dia")}
def celula(r):
    divisor, unidade = escala[r.indicador]
    casas = 3 if r.indicador == "indice_vale" else 1
    marca = " ✔" if r.robusto else (" (n.s.)" if r.p_valor >= 0.05 else " ~")
    return f"{r.coef / divisor:+.{casas}f}{unidade}{marca}"
c25["resultado"] = c25.apply(celula, axis=1)
sintese = c25.pivot_table(index="nivel", columns="indicador", values="resultado", aggfunc="first").loc[
    ["SIN", "SE", "S", "NE", "N"], list(escala)]
sintese.to_csv(PASTA_TABELAS / "05_sintese_2025_vs_2023.csv")
sintese

indicador,indice_vale,carga_residual_min,rampa_tarde,maior_rampa_horaria,horas_residual_baixa
nivel,,,,,
SIN,-0.031 ✔,-8.6 GW ✔,+9.7 GW ✔,+4.5 GW/h ✔,+3.2 h/dia ✔
SE,-0.025 ✔,-6.9 GW ✔,+8.6 GW ✔,+2.9 GW/h ✔,+3.5 h/dia ✔
S,-0.040 ✔,-1.1 GW ✔,+2.3 GW ✔,+0.5 GW/h ✔,-0.2 h/dia (n.s.)
NE,-0.057 ✔,-1.8 GW ✔,-1.1 GW ~,+0.4 GW/h ~,+3.1 h/dia ✔
N,+0.001 (n.s.),+0.2 GW ✔,+1.0 GW ✔,+0.1 GW/h ✔,-2.6 h/dia ✔


✔ = mesmo sinal e p < 0,05 nos 6 cenários de sensibilidade · ~ = mesmo sinal, mas não robusto em todos · (n.s.) = não significativo. Recorte: maio a dezembro, sem dias atípicos; ajuste por mês e dia útil; erros HAC.

## 3. Resposta objetiva

Entre 2023 e 2025, o vale do meio-dia **aprofundou** e a rampa do fim da tarde **cresceu**. No SIN, comparando dias do mesmo mês e tipo, a carga residual mínima entre 10h e 15h caiu **8,6 GW (−18%)**, a rampa até o início da noite subiu **9,7 GW (+45%)**, a maior subida em uma hora subiu **4,5 GW/h (+93%)** e as horas diárias de carga residual baixa passaram de **1,8 para 4,9**. As mudanças foram mais intensas no **Sudeste/Centro-Oeste** e no **Nordeste**, de **julho a outubro**, entre **10h e 14h** e **17h e 20h**, e continuaram de 2024 para 2025. São padrões **compatíveis** com a expansão da geração solar centralizada e distribuída; a base não mede a MMGD nem os cortes de geração, e o desenho não demonstra causalidade.

## 4. Recomendações

| # | Recomendação | Evidência | Ação |
|---|---|---|---|
| 1 | Priorizar as janelas críticas | Fig. 2 e 4 | SE e NE; julho a outubro; 10h–14h e 17h–20h como recorte padrão |
| 2 | Avaliar armazenamento | Fig. 1, 3 e 5 | Carregar entre 10h e 14h, descarregar entre 17h e 20h; dimensionar por região |
| 3 | Avaliar o intercâmbio NE–SE | Fig. 6 e A3 | Cruzar janelas com limites de intercâmbio e novas linhas; risco de corte no NE |
| 4 | Estudar resposta da demanda | Fig. 1 | Deslocar consumo industrial e comercial para o meio-dia; sinais tarifários |
| 5 | Integrar cortes e MMGD | Limitação 3.3 | Constrained-off e estimativa de MMGD do ONS |

Explicação detalhada (o que é, impacto, situação no Brasil, o que seria preciso): `docs/RECOMENDACOES_EXPLICADAS.md`.

## 5. Limitações

- A base não mede MMGD nem curtailment; os padrões são compatíveis com a expansão solar, mas não a demonstram.
- Desenho observacional: sem controle de temperatura, hidrologia, atividade econômica e tarifas.
- Três anos: comparação entre anos, não tendência de longo prazo.
- Quebra da série solar em 29/04/2023 (recorte de maio a dezembro); 5 dias atípicos excluídos (sensibilidade feita).
- Agregação por subsistema esconde restrições locais de transmissão.

## 6. Conferência dos entregáveis

In [ ]:
entregaveis = {
    "Relatório de uma folha (.docx)": RAIZ / "entrega" / "JeandersonMingoranca_RelatorioUmaFolha.docx",
    "Roteiro (PDF)": RAIZ / "docs" / "JeandersonMingoranca_ProjetoIntegrador_Roteiro.pdf",
    "Recomendações explicadas": RAIZ / "docs" / "RECOMENDACOES_EXPLICADAS.md",
    "Catálogo de figuras": PASTA_FIGURAS / "README.md",
    "Figura 1": PASTA_FIGURAS / "03_fig1_perfil_horario_sin.png",
    "Figura 5": PASTA_FIGURAS / "04_fig5_diferencas_ajustadas.png",
}
pd.DataFrame({"existe": {k: v.exists() for k, v in entregaveis.items()}})

,existe
Relatório de uma folha (.docx),True
Roteiro (PDF),True
Recomendações explicadas,False
Catálogo de figuras,True
Figura 1,True
Figura 5,True


: 